# Paper-Based IMC Workflow

## Step 4: Run Steinbock + MESMER segmentation and inspect masks

This notebook is the first **execution** step in the paper-based workflow.

In Step 3, we prepared a Steinbock-style project structure and a panel file with a `deepcell` grouping column. In this step, we actually use that project setup to run segmentation.

This is an important transition point because the segmentation masks produced here will become the basis for all later object measurement and downstream biological interpretation.

So this notebook is designed to do only two things:

1. run the Steinbock / DeepCell MESMER segmentation command
2. inspect the generated masks and stop there for quality review

We will **not** move on to feature extraction or quantification in this notebook.

## Why this step matters

Everything downstream depends on segmentation quality.

If the masks are poor, then:

- object measurements will be unreliable
- marker intensities will be assigned to the wrong cells
- phenotype labels will be distorted
- spatial analysis will be misleading

That is why we stop after segmentation and review the masks before proceeding.

## Official segmentation command we are following

From the Steinbock documentation, whole-cell MESMER segmentation is run as:

`steinbock segment deepcell --minmax`

The `--minmax` option applies channel-wise normalization for DeepCell input. Because our panel file contains a `deepcell` column, Steinbock will automatically:

- group `DNA1` and `DNA2` into the nuclear channel
- group `CD3`, `CD138`, `CD31`, and `aSMA` into the membrane channel
- ignore ungrouped channels for MESMER input

For reproducibility, we use a **pinned Steinbock Docker image version** rather than an unversioned latest tag.

## What this notebook will do technically

This notebook will:

1. point to the Steinbock project prepared in Step 3
2. construct a Docker command that runs the pinned Steinbock image
3. optionally execute that command
4. capture stdout/stderr to log files
5. list the produced mask files
6. visualize one example mask for review

To make this safe, execution is controlled by a notebook flag. That means you can inspect the command first, then choose when to actually run it.

In [ ]:
from pathlib import Path
import subprocess
import shutil
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt

try:
    from PIL import Image
except ImportError as exc:
    raise RuntimeError('Pillow is required for this notebook step.') from exc

WORKFLOW_ROOT = Path('/Users/rashid/1_IMC_Analysis/11_Vincenzo/paper_based_workflow')
PROJECT_ROOT = WORKFLOW_ROOT / 'step3_steinbock_project' / 'ROI001_D13'
IMAGES_DIR = PROJECT_ROOT / 'images'
MASKS_DIR = PROJECT_ROOT / 'masks'
PANEL_PATH = PROJECT_ROOT / 'panel.csv'
RUN_LOG = PROJECT_ROOT / 'step4_segmentation_stdout.log'
ERR_LOG = PROJECT_ROOT / 'step4_segmentation_stderr.log'
FIGURE_DIR = WORKFLOW_ROOT / 'step4_outputs' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

STEINBOCK_IMAGE = 'ghcr.io/bodenmillergroup/steinbock:0.16.0'
RUN_SEGMENTATION = False

print('Project root:', PROJECT_ROOT)
print('Images dir exists:', IMAGES_DIR.exists())
print('Panel exists:', PANEL_PATH.exists())
print('Masks dir:', MASKS_DIR)
print('Docker executable:', shutil.which('docker'))
print('Steinbock image:', STEINBOCK_IMAGE)
print('RUN_SEGMENTATION:', RUN_SEGMENTATION)


## Building the Docker command

We use Docker because Steinbock is distributed as a containerized workflow and your machine already has Docker available.

The command below does the following:

- mounts the Steinbock project root into the container as `/data`
- sets `/data` as the working directory inside the container
- runs the pinned Steinbock image version
- executes `steinbock segment deepcell --minmax`

This means Steinbock will read:

- `images/` for the channel TIFF files
- `panel.csv` for the `deepcell` grouping instructions

and then write segmentation masks into `masks/`.

In [ ]:
docker_command = [
    'docker', 'run', '--rm',
    '-v', f'{PROJECT_ROOT}:/data',
    '-w', '/data',
    STEINBOCK_IMAGE,
    'steinbock', 'segment', 'deepcell', '--minmax',
]

print('Docker command:')
print(' '.join(docker_command))


## Execution control

The next code cell is intentionally guarded by the `RUN_SEGMENTATION` flag.

- If `RUN_SEGMENTATION = False`, the notebook will only print a message and skip execution.
- If `RUN_SEGMENTATION = True`, the notebook will run the command and capture the logs.

This makes the execution boundary explicit, which is especially helpful in collaborative or teaching settings.

In [ ]:
if RUN_SEGMENTATION:
    start_time = datetime.now()
    print(f'Starting segmentation at {start_time.isoformat()}')
    result = subprocess.run(
        docker_command,
        capture_output=True,
        text=True,
        check=False,
    )
    RUN_LOG.write_text(result.stdout, encoding='utf-8')
    ERR_LOG.write_text(result.stderr, encoding='utf-8')
    end_time = datetime.now()
    print(f'Finished at {end_time.isoformat()}')
    print('Return code:', result.returncode)
    print('Stdout log:', RUN_LOG)
    print('Stderr log:', ERR_LOG)
else:
    print('Segmentation not executed. Set RUN_SEGMENTATION = True to run this step.')


## Inspecting mask outputs

If segmentation succeeded, Steinbock should create grayscale mask TIFFs in the `masks/` directory.

At this point, we are looking for very basic evidence that the run behaved correctly:

- were mask files created?
- do they match the expected image dimensions?
- do they contain multiple object IDs rather than a blank image?

This does not yet tell us whether the segmentation is biologically good, but it tells us whether the execution pipeline produced structurally valid outputs.

In [ ]:
mask_files = sorted(list(MASKS_DIR.glob('*.tif')) + list(MASKS_DIR.glob('*.tiff')))
print('Number of mask files found:', len(mask_files))
for path in mask_files[:10]:
    print('-', path.name)


## First-pass mask quality inspection

For review, we visualize one mask file and summarize a few simple properties:

- image shape
- data type
- number of unique values
- maximum object label

A valid object mask should usually contain:

- background encoded as `0`
- many positive integer object IDs
- a reasonable number of unique labels

If the mask is nearly empty or contains only one or two values, that is a warning sign.

In [ ]:
if mask_files:
    example_mask_path = mask_files[0]
    with Image.open(example_mask_path) as img:
        mask = np.array(img)
    unique_vals = np.unique(mask)
    print('Example mask:', example_mask_path.name)
    print('Shape:', mask.shape)
    print('Dtype:', mask.dtype)
    print('Number of unique values:', len(unique_vals))
    print('Max label:', int(mask.max()))
else:
    print('No mask files available yet.')


In [ ]:
if mask_files:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(mask, cmap='gray')
    axes[0].set_title('Mask as Grayscale')
    axes[0].axis('off')

    display_mask = mask.astype(np.float32)
    if display_mask.max() > 0:
        display_mask = display_mask / display_mask.max()
    axes[1].imshow(display_mask, cmap='nipy_spectral')
    axes[1].set_title('Mask with Pseudo-Color Labels')
    axes[1].axis('off')

    plt.tight_layout()
    figure_path = FIGURE_DIR / 'step4_example_mask_review.png'
    plt.savefig(figure_path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print('Saved review figure:', figure_path)
else:
    print('Skipping figure generation because no masks were found.')


## How to review Step 4

Once you run this notebook, look for the following:

### 1. Did the Docker command complete successfully?
Check the return code and the log files. A return code of `0` is a good sign, but it is still worth scanning stderr/stdout for warnings.

### 2. Were mask files created?
If `masks/` is empty, then segmentation did not actually complete successfully.

### 3. Do the masks look structurally plausible?
We do not need perfection yet, but we do need evidence that objects are being separated rather than merged into a single mass or missed entirely.

### 4. Why we stop here
If the masks are poor, it is better to correct the segmentation inputs now than to push bad masks into quantification.

## Stop point

This notebook stops after segmentation execution and first-pass mask inspection.

If the masks look acceptable, the next step will be to run Steinbock object measurement and start building the single-cell table.